# 🔍 RAG Pipeline Baseline
Pipeline: **Query Rewrite → Query Classifier → Hybrid Search (BM25 + Semantic + RRF) → Rerank → Context Combiner → Context Builder**

## 1. Setup & Config

In [26]:
import sys
import os
import time
import json
from pathlib import Path

# Project root
PROJECT_DIR = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_DIR))

print(f"📁 Project dir: {PROJECT_DIR}")

# Data paths
CHUNKS_PATH = str(PROJECT_DIR / "data" / "rag_chunks_v2.json")
EMBEDDINGS_PATH = str(PROJECT_DIR / "data" / "embeddings.npy")

print(f"📄 Chunks: {CHUNKS_PATH}")
print(f"📄 Embeddings: {EMBEDDINGS_PATH}")

from dotenv import load_dotenv
load_dotenv(PROJECT_DIR / ".env")
print(f"🔑 API Key loaded: {'✅' if os.getenv('GENAI_API_KEY') else '❌'}")

📁 Project dir: c:\Users\Admin\OneDrive - Hanoi University of Science and Technology\Desktop\DATN
📄 Chunks: c:\Users\Admin\OneDrive - Hanoi University of Science and Technology\Desktop\DATN\data\rag_chunks_v2.json
📄 Embeddings: c:\Users\Admin\OneDrive - Hanoi University of Science and Technology\Desktop\DATN\data\embeddings.npy
🔑 API Key loaded: ✅


## 2. Khởi tạo Components

In [27]:
from src.rag.retrieve_rebuild import CustomSearch
from src.rag.reranker import Reranker
from src.llm.query_rewriter import QueryRewriter
from src.rag.adaptive_rag import AdaptiveRAGAgent
from src.rag.context_builder import ContextBuilder
from src.config.config import settings

print("[Node 0] Initializing Retriever (CustomSearch)...")
t0 = time.time()
retriever = CustomSearch(chunks_path=CHUNKS_PATH, embeddings_path=EMBEDDINGS_PATH)
print(f"✅ Retriever init xong ({time.time()-t0:.2f}s)")

print("\n[Node 0] Initializing Reranker...")
reranker = Reranker()
print("✅ Reranker init xong (lazy load)")

print("\n[Node 0] Initializing QueryRewriter...")
rewriter = QueryRewriter()
print("✅ QueryRewriter init xong")

print("\n[Node 0] Initializing AdaptiveRAGAgent...")
rag_agent = AdaptiveRAGAgent(retriever=retriever, reranker=reranker, settings=settings)
print("✅ AdaptiveRAGAgent init xong")

print("\n[Node 0] Initializing ContextBuilder...")
context_builder = ContextBuilder()
print("✅ ContextBuilder init xong")

[Node 0] Initializing Retriever (CustomSearch)...
CustomSearch initialized: 2348 docs, vocab=9672, avgdl=137.7
✅ Retriever init xong (18.93s)

[Node 0] Initializing Reranker...
✅ Reranker init xong (lazy load)

[Node 0] Initializing QueryRewriter...
✅ QueryRewriter init xong

[Node 0] Initializing AdaptiveRAGAgent...
✅ AdaptiveRAGAgent init xong

[Node 0] Initializing ContextBuilder...
✅ ContextBuilder init xong


## 3. Khởi tạo Test Case

In [28]:
query = "tổng hợp kiến thức chương A của quyển Cánh Diều đi"
history_context = ""

print(f"Input Query: '{query}'")
print(f"History Context: '{history_context}'")

Input Query: 'tổng hợp kiến thức chương A của quyển Cánh Diều đi'
History Context: ''


## 4. Thực thi từng Node

### NODE 1: Query Rewriter

In [29]:
print("🟢 NODE 1: Query Rewriter")
t0 = time.time()
rewritten_queries = rewriter.rewrite(query, history_context)
print(f"\n=> Output Queries ({time.time()-t0:.2f}s):")
print(json.dumps(rewritten_queries, indent=2, ensure_ascii=False))

if not rewritten_queries:
    rewritten_queries = [query]
    
primary_query = rewritten_queries[0]

🟢 NODE 1: Query Rewriter

=> Output Queries (1.54s):
[
  "Tổng hợp kiến thức chương A sách giáo khoa Tin học lớp 10 Cánh Diều",
  "Tóm tắt kiến thức chương A sách Tin học 10 Cánh Diều",
  "Các nội dung chính của chương A sách Tin học 10 Cánh Diều"
]


### NODE 2: Query Classifier (Strategy Selection)

In [30]:
import dataclasses
print("🟢 NODE 2: Query Classifier (Strategy Selection)")

profile = rag_agent.classifier.classify(primary_query)
print(f"\n=> Output Profile (Schema: QueryProfile):")
profile_dict = dataclasses.asdict(profile)
# Convert Enum to string for JSON serialization
profile_dict['strategy'] = profile_dict['strategy'].value
print(json.dumps(profile_dict, indent=2, ensure_ascii=False))

🟢 NODE 2: Query Classifier (Strategy Selection)

=> Output Profile (Schema: QueryProfile):
{
  "strategy": "broad",
  "grade": "10",
  "topic_hint": null,
  "top_k_override": null,
  "reason": "Query tổng quát: broad=True, grade_only=True, topic_broad=False"
}


### NODE 3: Hybrid Search (Retriever)

In [31]:
from src.schemas.rag_outputs import RetrievalOutput, RetrievalResult

print(f"🟢 NODE 3: Hybrid Search ({profile.strategy.value} strategy)")
t0 = time.time()

# Lấy top chunks dựa trên strategy
raw_chunks = []
if profile.strategy.value == "curriculum":
    raw_chunks = rag_agent._curriculum_lookup(profile)
elif profile.strategy.value == "broad":
    raw_chunks = rag_agent._broad_retrieval(primary_query, profile)
    if len(raw_chunks) < 3:
        standard = retriever.search(primary_query, top_k=settings.RETRIEVER_TOP_K)
        raw_chunks = rag_agent._merge_deduplicate(raw_chunks, standard)
elif profile.strategy.value == "hierarchical":
    raw_chunks = rag_agent._hierarchical_retrieval(primary_query, profile)
else:
    raw_chunks = retriever.search(primary_query, top_k=20, top_n=30)
    
print(f"\n=> Retrieved {len(raw_chunks)} chunks ({time.time()-t0:.2f}s).")

# Convert sang Schema RetrievalOutput
retrieval_results = []
for r in raw_chunks:
    retrieval_results.append(RetrievalResult(
        content=r["content"],
        context=r.get("context"),
        metadata=r.get("metadata", {}),
        bm25_score=None,
        faiss_score=None,
        rrf_score=r.get("score")
    ))

retrieval_output = RetrievalOutput(results=retrieval_results, query=primary_query)

print("\n=> Output Schema (RetrievalOutput):")
# In ra raw json của schema (rút gọn nếu quá dài)
schema_json = retrieval_output.model_dump_json(indent=2)
print(schema_json[:])

🟢 NODE 3: Hybrid Search (broad strategy)

=> Retrieved 30 chunks (0.00s).

=> Output Schema (RetrievalOutput):
{
  "results": [
    {
      "context": "Máy tính và xã hội tri thức – Tin học và xử lí thông tin > DỮ LIỆU, THÔNG TIN VÀ XỬ LÍ THÔNG TIN > Mục tiêu bài học",
      "content": "Học xong bài này, em sẽ:\n*   Biết được thông tin là gì, dữ liệu là gì.\n*   Phân biệt được thông tin và dữ liệu, nêu được ví dụ minh họa.\n*   Biết được xử lí thông tin là gì.",
      "metadata": {
        "grade": "10",
        "lesson": "Bài 1",
        "idea": null,
        "level": 2,
        "title": "Mục tiêu bài học",
        "type": "objective"
      },
      "bm25_score": null,
      "faiss_score": null,
      "rrf_score": 1.0
    },
    {
      "context": "Máy tính và xã hội tri thức – Tin học và xử lí thông tin > SỰ ƯU VIỆT CỦA MÁY TÍNH VÀ NHỮNG THÀNH TỰU CỦA TIN HỌC > Mục tiêu bài học",
      "content": "Học xong bài này, em sẽ:\n* Nêu được sự ưu việt của việc lưu trữ, xử lí và truyền thông

### NODE 4: Reranker

In [32]:
from src.schemas.rag_outputs import RerankOutput, RerankResult

print("🟢 NODE 4: Reranker")
t0 = time.time()

# Rerank và filter
if profile.strategy.value not in ["curriculum"]:
    rerank_top_n = getattr(settings, 'RERANKER_TOP_N', 5)
    reranked_chunks = reranker.rerank(primary_query, raw_chunks, top_n=rerank_top_n)
    
    min_score = getattr(settings, 'RERANKER_MIN_SCORE', 0.15)
    filtered_chunks = reranker.filter_context(reranked_chunks, min_score=min_score)
    
    if not filtered_chunks and reranked_chunks:
        filtered_chunks = reranked_chunks[:3] # fallback
else:
    filtered_chunks = raw_chunks
    
print(f"\n=> Output sau Reranking & Filter ({time.time()-t0:.2f}s): {len(filtered_chunks)} chunks.")

# Convert sang Schema RerankOutput
rerank_results = []
for r in filtered_chunks:
    rerank_results.append(RerankResult(
        content=r["content"],
        context=r.get("context"),
        metadata=r.get("metadata", {}),
        rrf_score=r.get("score"),
        rerank_score=r.get("rerank_score", r.get("score", 1.0))
    ))

rerank_output = RerankOutput(results=rerank_results, query=primary_query, top_n=len(rerank_results))

print("\n=> Output Schema (RerankOutput):")
schema_json_rerank = rerank_output.model_dump_json(indent=2)
print(schema_json_rerank[:])

🟢 NODE 4: Reranker
Loading reranker: AITeamVN/Vietnamese_Reranker...
Reranker loaded on cuda

=> Output sau Reranking & Filter (10.75s): 5 chunks.

=> Output Schema (RerankOutput):
{
  "results": [
    {
      "context": "Máy tính và xã hội tri thức – CS: Biểu diễn thông tin > HỆ NHỊ PHÂN VÀ ỨNG DỤNG > Học xong bài này, em sẽ:",
      "content": "*   Hiểu và thực hiện được các phép toán cơ bản NOT, AND, OR và XOR theo từng bit và cho các dãy bit.\n*   Biết **hệ nhị phân** (hệ đếm cơ số 2) là gì.\n*   Chuyển đổi được số đếm hệ nhị phân sang giá trị thập phân và ngược lại.\n*   Biết được các **phép toán bit** là cơ sở để thực hiện các tính toán số học nhị phân.\n*   Giải thích được ứng dụng của hệ nhị phân trong tin học.",
      "metadata": {
        "grade": "10",
        "lesson": "Bài 1",
        "idea": null,
        "level": 2,
        "title": "Học xong bài này, em sẽ:",
        "type": "objective"
      },
      "rrf_score": 1.0,
      "rerank_score": 0.0017291937256231904,
      

### NODE 5: Context Combiner

In [33]:
from src.rag.context_combiner import format_contexts

print("🟢 NODE 5: Context Combiner (Format)")

# Dùng lại list dict cho hàm format_contexts
schema_chunks_dict = [r.model_dump() for r in rerank_output.results]

# Test mode flat
flat_context = format_contexts(schema_chunks_dict, action="chat")
print(f"Flat Context Length: {len(flat_context)} ký tự")
print(f"Flat Context Preview:\n{flat_context[:250]}...\n")

# Test mode grouped (dùng cho slide/lesson_plan)
grouped_context = format_contexts(schema_chunks_dict, action="slide")
print(f"Grouped Context Length: {len(grouped_context)} ký tự")
print(f"Grouped Context Preview:\n{grouped_context[:]}...\n")

🟢 NODE 5: Context Combiner (Format)
Flat Context Length: 2128 ký tự
Flat Context Preview:
Context 1:
*   Hiểu và thực hiện được các phép toán cơ bản NOT, AND, OR và XOR theo từng bit và cho các dãy bit.
*   Biết **hệ nhị phân** (hệ đếm cơ số 2) là gì.
*   Chuyển đổi được số đếm hệ nhị phân sang giá trị thập phân và ngược lại.
*   Biết đượ...

Grouped Context Length: 2217 ký tự
Grouped Context Preview:
## Chủ đề: Khác

### Bài: Khác

Context 1 [score=0.0017]:
*   Hiểu và thực hiện được các phép toán cơ bản NOT, AND, OR và XOR theo từng bit và cho các dãy bit.
*   Biết **hệ nhị phân** (hệ đếm cơ số 2) là gì.
*   Chuyển đổi được số đếm hệ nhị phân sang giá trị thập phân và ngược lại.
*   Biết được các **phép toán bit** là cơ sở để thực hiện các tính toán số học nhị phân.
*   Giải thích được ứng dụng của hệ nhị phân trong tin học.

Context 2 [score=0.0005]:
*   Trình bày được những đóng góp cơ bản của tin học đối với xã hội, nếu được ví dụ minh hoạ.
*   Nhận biết được một vài thiết bị số t

### NODE 6: Context Builder (Synthesis via LLM)

In [34]:
print("🟢 NODE 6: Context Builder (LLM Synthesis)")
t0 = time.time()

# Truyền vào top 5 chunks để build context tổng hợp
action_for_builder = "slide" if profile.strategy.value == "curriculum" else "chat"
synthesized_context = context_builder.build(
    query=primary_query, 
    chunks=schema_chunks_dict[:5], 
    action=action_for_builder
)

print(f"\n=> Synthesized Context ({time.time()-t0:.2f}s) Length: {len(synthesized_context)} ký tự")
print(f"Content Preview:\n{synthesized_context}")

🟢 NODE 6: Context Builder (LLM Synthesis)

=> Synthesized Context (3.61s) Length: 2478 ký tự
Content Preview:
## Tổng hợp kiến thức Chương A - Tin học lớp 10 (Sách Cánh Diều)

Chương A của sách giáo khoa Tin học lớp 10 (Cánh Diều) tập trung vào các kiến thức nền tảng về tin học, bao gồm khái niệm cơ bản, vai trò của tin học trong xã hội và các công nghệ liên quan.

### 1. Khái niệm cơ bản về Thông tin và Dữ liệu

*   **Thông tin:** Là những gì mang lại sự hiểu biết, làm rõ ý nghĩa, làm giảm sự không chắc chắn.
*   **Dữ liệu:** Là thông tin đã được biểu diễn dưới một dạng thức nào đó để máy tính có thể xử lý.
*   **Xử lý thông tin:** Là quá trình thu thập, lưu trữ, tìm kiếm, sắp xếp, phân tích, biến đổi và truyền tin.

### 2. Hệ nhị phân và Phép toán Bit

*   **Hệ nhị phân (hệ đếm cơ số 2):** Là hệ đếm chỉ sử dụng hai ký hiệu là 0 và 1.
*   **Phép toán bit:** Bao gồm các phép toán cơ bản như NOT, AND, OR, XOR, được thực hiện theo từng bit hoặc cho các dãy bit.
*   **Ứng dụng:** Các phép